# 01 Reconcile: Kaggle `ipl_cricket` vs Cricsheet `crickql.raw`

Before anything downstream trusts the new pipeline, compare it with the load I already had.
Three levels, each one narrower than the last:

1. **Season:** matches, deliveries and runs per season
2. **Match:** which match IDs exist on only one side
3. **Delivery:** for matches on both sides, do delivery counts and runs agree?

Every gap gets an explanation in the findings cell at the bottom. A gap with no explanation is a bug until proven otherwise.

Kaggle columns used: `matches(id, season, date)`, `deliveries(match_id, inning, total_runs)`.

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")
BASE = dict(host=os.getenv("PG_HOST", "localhost"), port=int(os.getenv("PG_PORT", "5432")),
            username=os.getenv("PG_USER") or None, password=os.getenv("PG_PASSWORD") or None)
KAGGLE_DB = "ipl_cricket"
CRICKQL_DB = os.getenv("PG_DB", "crickql")


ENGINES = {}


def q(db, sql):
    if db not in ENGINES:
        ENGINES[db] = create_engine(URL.create("postgresql+psycopg2", database=db, **BASE))
    with ENGINES[db].connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Season level

In [2]:
kaggle_season = q(KAGGLE_DB, """
    SELECT m.season::text AS season,
           count(DISTINCT m.id)  AS k_matches,
           count(d.match_id)     AS k_deliveries,
           sum(d.total_runs)     AS k_runs
    FROM matches m LEFT JOIN deliveries d ON d.match_id = m.id
    GROUP BY 1
""")
cricsheet_season = q(CRICKQL_DB, """
    SELECT m.season_raw AS season, min(m.season_year) AS season_year,
           count(DISTINCT m.match_id) AS c_matches,
           count(d.match_id)          AS c_deliveries,
           sum(d.runs_total)          AS c_runs
    FROM raw.matches m LEFT JOIN raw.deliveries d USING (match_id)
    GROUP BY 1
""")
season = cricsheet_season.merge(kaggle_season, on="season", how="outer").sort_values("season")
for col in ("matches", "deliveries", "runs"):
    season[f"diff_{col}"] = season[f"c_{col}"].fillna(0) - season[f"k_{col}"].fillna(0)
season

,season,season_year,c_matches,c_deliveries,c_runs,k_matches,k_deliveries,k_runs,diff_matches,diff_deliveries,diff_runs
0,2007/08,2008,58,13489,17937,58.0,13489.0,17937.0,0.0,0.0,0.0
1,2009,2009,57,13606,16353,57.0,13606.0,16353.0,0.0,0.0,0.0
2,2009/10,2010,60,14498,18883,60.0,14498.0,18883.0,0.0,0.0,0.0
3,2011,2011,73,17013,21154,73.0,17013.0,21154.0,0.0,0.0,0.0
4,2012,2012,74,17767,22453,74.0,17767.0,22453.0,0.0,0.0,0.0
5,2013,2013,76,18177,22602,76.0,18177.0,22602.0,0.0,0.0,0.0
6,2014,2014,60,14300,18931,60.0,14300.0,18931.0,0.0,0.0,0.0
7,2015,2015,59,13652,18353,59.0,13652.0,18353.0,0.0,0.0,0.0
8,2016,2016,60,14096,18862,60.0,14096.0,18862.0,0.0,0.0,0.0
9,2017,2017,59,13862,18786,59.0,13862.0,18786.0,0.0,0.0,0.0


In [3]:
season_gaps = season[(season[["diff_matches", "diff_deliveries", "diff_runs"]] != 0).any(axis=1)]
season_gaps

,season,season_year,c_matches,c_deliveries,c_runs,k_matches,k_deliveries,k_runs,diff_matches,diff_deliveries,diff_runs
17,2025,2025,74,17285,26527,NaN,NaN,NaN,74.0,17285.0,26527.0
18,2026,2026,74,17527,27455,NaN,NaN,NaN,74.0,17527.0,27455.0


## 2. Match level

In [4]:
k_ids = set(q(KAGGLE_DB, "SELECT id FROM matches")["id"])
c = q(CRICKQL_DB, "SELECT match_id, season_raw, match_date, result_type FROM raw.matches")
c_ids = set(c["match_id"])
only_cricsheet = c[c["match_id"].isin(c_ids - k_ids)]
only_kaggle = sorted(k_ids - c_ids)
print(f"both: {len(k_ids & c_ids)}  only Cricsheet: {len(only_cricsheet)}  "
      f"only Kaggle: {len(only_kaggle)}")
only_cricsheet.groupby("season_raw").size().rename("matches_only_in_cricsheet")

both: 1095  only Cricsheet: 148  only Kaggle: 0


season_raw
2025    74
2026    74
Name: matches_only_in_cricsheet, dtype: int64

In [5]:
# Kaggle matches Cricsheet does not have (should be empty; anything here needs a reason)
only_kaggle[:20]

[]

## 3. Delivery level, matches on both sides

In [6]:
k_match = q(KAGGLE_DB, """
    SELECT match_id, count(*) AS k_deliveries, sum(total_runs) AS k_runs,
           count(*) FILTER (WHERE inning > 2) AS k_super_over_balls
    FROM deliveries GROUP BY 1
""")
c_match = q(CRICKQL_DB, """
    SELECT match_id, count(*) AS c_deliveries, sum(runs_total) AS c_runs,
           count(*) FILTER (WHERE is_super_over) AS c_super_over_balls
    FROM raw.deliveries GROUP BY 1
""")
both = c_match.merge(k_match, on="match_id", how="inner")
mismatch = both[(both.c_deliveries != both.k_deliveries) | (both.c_runs != both.k_runs)]
print(f"matches compared: {len(both)}, mismatched: {len(mismatch)}")
mismatch.head(20)

matches compared: 1095, mismatched: 0


,match_id,c_deliveries,c_runs,c_super_over_balls,k_deliveries,k_runs,k_super_over_balls


In [7]:
# Numbers for the findings cell
total_c = q(CRICKQL_DB, "SELECT (SELECT count(*) FROM raw.matches) m, "
                        "(SELECT count(*) FROM raw.deliveries) d")
total_k = q(KAGGLE_DB, "SELECT (SELECT count(*) FROM matches) m, "
                       "(SELECT count(*) FROM deliveries) d")
print("Cricsheet:", total_c.iloc[0].to_dict())
print("Kaggle:   ", total_k.iloc[0].to_dict())
print("Seasons with gaps:", season_gaps["season"].tolist())
print("Super-over balls, Kaggle vs Cricsheet:",
      int(both.k_super_over_balls.sum()), int(both.c_super_over_balls.sum()))

Cricsheet: {'m': 1243, 'd': 295732}
Kaggle:    {'m': 1095, 'd': 260920}
Seasons with gaps: ['2025', '2026']
Super-over balls, Kaggle vs Cricsheet: 161 161


## Findings

- **Overall:** Cricsheet has 1,243 matches and 295,732 deliveries. Kaggle has 1,095 matches and 260,920 deliveries. The gap is 148 matches and 34,812 deliveries, and all of it sits in two seasons.
- **Seasons only in Cricsheet:** 2025 (74 matches, 17,285 deliveries, 26,527 runs) and 2026 (74 matches, 17,527 deliveries, 27,455 runs). Together that is 148 matches and 34,812 deliveries, the full gap. Kaggle's dataset ends at 2024. All 17 seasons from 2008 to 2024 show zero difference on matches, deliveries and runs.
- **Matches only in Kaggle:** 0. All 1,095 Kaggle match IDs exist in Cricsheet. The Kaggle data is a subset of the same source, not a different one.
- **Delivery-level mismatches:** 0 of 1,095 shared matches differ on delivery count or total runs. The new parser reproduces the old load ball for ball.
- **Super overs:** 161 super-over balls on both sides. Kaggle keeps them in the deliveries table as inning 3 and above, so any Kaggle query that summed runs without filtering them out was overcounting. CrickQL flags them with `is_super_over` and excludes them from the marts by default.
- **Verdict:** Yes. The two loads agree exactly on every shared match, and Cricsheet adds two seasons Kaggle doesn't have. The Cricsheet pipeline replaces the Kaggle load with no loss of history.
  